In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# =====================================================================
# 1. 데이터셋 활용 및 삭제 (Drop)
# =====================================================================
df = pd.read_csv('Clean_Dataset.csv')

# [피드백 1번 반영] 과적합을 유발하는 편명(flight)과 불필요한 인덱스 삭제
columns_to_drop = []
if 'Unnamed: 0' in df.columns:
    columns_to_drop.append('Unnamed: 0')
if 'flight' in df.columns:
    columns_to_drop.append('flight')
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# [피드백 3번 반영] 오류가 있던 season(성수기/비수기) 로직은 생성하지 않고 완전히 삭제함


# =====================================================================
# 3. 변수 생성 (Feature Engineering)
# =====================================================================
# ① 노선(Route) 변수 생성: 출발지_도착지
df['route'] = df['source_city'] + "_" + df['destination_city']
# route 생성 후 원래 도시 컬럼은 제거
df.drop(columns=['source_city', 'destination_city'], inplace=True)

# ② 예약 시점 구간 변수 생성
# [피드백 2번 반영] -1부터 시작하여 0일 데이터 손실(결측치) 완벽 방지
bins = [-1, 7, 21, df['days_left'].max()]
labels = ['1~7일 전(출발 직전)', '8~21일 전(단기)', '22일 이상(장기)']
df['booking_period'] = pd.cut(df['days_left'], bins=bins, labels=labels)

# ③ 타겟 변수 로그 변환 (가격 변동성 학습 최적화)
df['price'] = np.log1p(df['price'])


# =====================================================================
# 2. 숫자로 변환 (Encoding)
# =====================================================================
# ① class (절대 지우지 않고 0과 1로 매핑)
df['class'] = df['class'].map({'Economy': 0, 'Business': 1})

# ② stops: 직항 0, 1회 경유 1, 2회 이상 2
df['stops'] = df['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})

# ③ 도시 이름 원-핫 인코딩 (6개 도시 분할)
df = pd.get_dummies(df, drop_first=True)


# =====================================================================
# 4. 데이터 분리 (Train/Test Split - 8:2 비율)
# =====================================================================
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== 최종 전처리 및 데이터 분리 완료 ===")
print(f"X_train 데이터 크기: {X_train.shape}")
print(f"y_train 데이터 크기: {y_train.shape}")
print(f"X_test 데이터 크기: {X_test.shape}")
print(f"y_test 데이터 크기: {y_test.shape}")
# =====================================================================
# 5. CSV 파일로 저장하기 (눈으로 확인하기 위한 용도)
# =====================================================================

# 1. 전체 전처리 완료된 데이터 하나로 저장하기
df.to_csv('final_preprocessed_data.csv', index=False, encoding='utf-8-sig')

# 2. (선택사항) 머신러닝용으로 분리된 Train / Test 데이터를 각각 저장하기
# 정답(y) 컬럼을 다시 붙여서 보기 좋게 하나의 파일로 만듭니다.
train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv('train_data.csv', index=False, encoding='utf-8-sig')

test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False, encoding='utf-8-sig')

print("CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.")

=== 최종 전처리 및 데이터 분리 완료 ===
X_train 데이터 크기: (240122, 50)
y_train 데이터 크기: (240122,)
X_test 데이터 크기: (60031, 50)
y_test 데이터 크기: (60031,)
CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ==============================
# 1. X, y 분리
# ==============================
X = df.drop(columns=['price'])
y = df['price']

# ==============================
# 2. Train / Test Split
# ==============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 3. Linear Regression 모델 학습
# ==============================
model = LinearRegression()
model.fit(X_train, y_train)

# ==============================
# 4. 예측
# ==============================
y_pred = model.predict(X_test)

# ==============================
# 5. 성능 평가
# ==============================
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===== 모델 성능 =====")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# ==============================
# 6. 계수 확인
# ==============================
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

print("\n===== 가격 상승 영향 Top 10 =====")
print(coef_df.head(10))

===== 모델 성능 =====
MAE: 0.2488758786171071
MSE: 0.10127304895510002
RMSE: 0.3182342674117607
R2: 0.9182320511933398

===== 가격 상승 영향 Top 10 =====
                    Feature  Coefficient
1                     class     2.037264
8           airline_Vistara     0.651858
4         airline_Air_India     0.524682
7          airline_SpiceJet     0.472856
5          airline_GO_FIRST     0.439275
0                     stops     0.373224
6            airline_Indigo     0.332216
40      route_Kolkata_Delhi     0.188158
38  route_Kolkata_Bangalore     0.182332
39    route_Kolkata_Chennai     0.167922


In [30]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# ==============================
# 1. X, y 분리
# ==============================
X = df.drop(columns=['price'])
y = df['price']

# ==============================
# 2. Train / Test Split
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 3. Lasso 모델 학습
# ==============================
model = Lasso(alpha=0.01)   # alpha 중요!
model.fit(X_train, y_train)

# ==============================
# 4. 예측
# ==============================
y_pred = model.predict(X_test)

# ==============================
# 5. 성능 평가
# ==============================
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===== Lasso 모델 성능 =====")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# ==============================
# 6. 계수 확인 (중요)
# ==============================
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

# 0 아닌 것만 보기 
coef_df = coef_df[coef_df['Coefficient'] != 0]

coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

print("\n===== Lasso 선택된 변수 (Top 10) =====")
print(coef_df.head(10))

===== Lasso 모델 성능 =====
MAE: 0.27518459178018395
MSE: 0.12393492045765139
RMSE: 0.3520439183648134
R2: 0.8999348362086772

===== Lasso 선택된 변수 (Top 10) =====
                   Feature  Coefficient
1                    class     2.027703
0                    stops     0.282029
8          airline_Vistara     0.220924
4        airline_Air_India     0.066045
2                 duration     0.011074
12  departure_time_Morning     0.009863
15    arrival_time_Evening     0.005979
3                days_left    -0.014507


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# ==============================
# 1. X, y 분리
# ==============================
X = df.drop(columns=['price'])
y = df['price']

# ==============================
# 2. Train / Test Split
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 3. Random Forest 모델 학습
# ==============================
model = RandomForestRegressor(
    n_estimators=100,     # 트리 개수
    max_depth=None,       # 깊이 제한 없음
    random_state=42,
    n_jobs=-1             # CPU 다 씀 (빠름)
)

model.fit(X_train, y_train)

# ==============================
# 4. 예측
# ==============================
y_pred_rf = model.predict(X_test)

# ==============================
# 5. 성능 평가
# ==============================
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)

print("===== Random Forest 모델 성능 =====")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# ==============================
# 6. Feature 중요도
# ==============================
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

print("\n===== 중요 변수 Top 10 =====")
print(feature_importance.head(10))